# Stuff- : الخط الهجين (Blender لعالم الـ3D + بطاقات ثنائية الأبعاد)
شغّل الخلايا بالترتيب. **الرندر يشغّله المستخدم فقط.**
- يلزم `Runtime > Change runtime type > T4 GPU`. Cycles يستعمل الكرت عبر CUDA/OptiX **دون حاجة لمكتبات OpenGL**.
- المدخل الوحيد: السكربت الإنجليزي في الخلية 3. ابدأ بمعاينة `FRAMES = "0-89"` (ثلاث ثوانٍ) قبل الفيديو الكامل.
- أي خطأ: ألصق لي ناتج الخلية التي فشلت كاملًا.

In [ ]:
# 1) Setup (safe to re-run; installs once)
import os
os.chdir("/content")
REPO = "/content/Stuff-"
def node_major():
    import subprocess
    try: return int(subprocess.check_output(["node", "-v"]).decode().strip()[1:].split(".")[0])
    except Exception: return 0
if not os.path.exists("/content/.hybrid_installed"):
    !rm -rf {REPO}
    !git clone -q https://github.com/marymyc/Stuff-.git {REPO}
    !apt-get -qq install -y ffmpeg fonts-dejavu-core libsndfile1 libxi6 libxxf86vm1 libxfixes3 libgl1 libxrender1 libsm6 libegl1 libxkbcommon0 libnss3 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libgbm1 libxrandr2 libxcomposite1 libxdamage1 libpango-1.0-0 libcairo2 libcups2 > /dev/null 2>&1
    !(apt-get -qq install -y libasound2t64 || apt-get -qq install -y libasound2) > /dev/null 2>&1
    if node_major() < 18:
        !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
        !apt-get install -y -qq nodejs > /dev/null
    !cd {REPO}/hybrid && npm install --no-audit --no-fund --loglevel=error 2>&1 | tail -3
    print("Installing Python packages (Chatterbox + torch 2.6, several minutes)...")
    !pip install --progress-bar off -q -e "{REPO}[tts]" 2>&1 | tail -n 3
    open("/content/.hybrid_installed", "w").write("ok")
else:
    !git -C {REPO} pull -q
!pip uninstall -y -q torchvision
os.chdir(REPO + "/hybrid")
print("ready:", os.getcwd(), "| node", node_major())

In [ ]:
# 2) Blender 5.2 (downloaded once) + GPU check
import glob, os, re, urllib.request
os.chdir("/content")
BASE = "https://download.blender.org/release/Blender5.2/"
def blender_path():
    found = glob.glob("/content/blender-5.2*-linux-x64/blender")
    return sorted(found, key=lambda p: [int(x) for x in re.findall(r"\d+", os.path.basename(os.path.dirname(p)))])[-1] if found else None
if not blender_path():
    html = urllib.request.urlopen(BASE).read().decode()
    files = sorted(set(re.findall(r"blender-5\.2\.\d+-linux-x64\.tar\.xz", html)), key=lambda f: [int(x) for x in re.findall(r"\d+", f)])
    assert files, "No Blender 5.2 linux tarball listed at " + BASE
    !wget -q {BASE}{files[-1]} -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content
BL = blender_path()
os.environ["BLENDER_BIN"] = BL
!{BL} --version | head -1
!nvidia-smi -L || echo "NO GPU: Runtime > Change runtime type > T4 GPU"
os.chdir("/content/Stuff-/hybrid")

In [ ]:
# 3) YOUR INPUT
SCRIPT = """Paste your English script here (about 150-170 words for 60-70 seconds)."""
NAME = "short1"
REFERENCE_WAV = ""      # optional: clean 10+ s American-male clip you have rights to
ENGINE = "CYCLES"       # GPU through CUDA/OptiX
SAMPLES = 12            # raise to 24-32 for a cleaner final render
SCALE = 1.0             # 1.0 = 1080x1920
FRAMES = "0-89"         # preview = first 3 seconds; set "" for the full video

import os
os.chdir("/content/Stuff-/hybrid")
open("/content/script.txt", "w").write(SCRIPT)
print(len(SCRIPT.split()), "words")

In [ ]:
# 4) Make the video (voice -> plan -> cards -> Blender -> captions + audio). Progress prints below.
import os, subprocess
os.chdir("/content/Stuff-/hybrid")
cmd = ["python", "tools/make.py", "/content/script.txt", "--name", NAME, "--engine", ENGINE, "--samples", str(SAMPLES), "--scale", str(SCALE)]
if REFERENCE_WAV: cmd += ["--ref-voice", REFERENCE_WAV]
if FRAMES: cmd += ["--frames", FRAMES]
p = subprocess.Popen(cmd, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
print("exit code:", p.wait())

In [ ]:
# 5) Download
from google.colab import files
files.download(f"/content/Stuff-/hybrid/out/{NAME}.mp4")